# 01 — Exploratory Data Analysis

**Research question:** Does ward-level colonization pressure (environmental exposure)
or antibiotic exposure history (patient exposure) better predict hospital-onset MRSA
acquisition?

This notebook loads `HO_infxn_analysis.csv`, filters to the MRSA subsets of both matched
cohorts (`environmental` and `patient`), fixes the `age` column's `'>90'` privacy-censoring
text value, and profiles both subsets before any modeling.

See [DATA_ACCESS.md](../DATA_ACCESS.md) for what the two `analysis` arms mean and how they
differ in matching strategy.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_raw, clean_age, save_processed

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## Load raw data and get oriented

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

In [ ]:
raw.dtypes.value_counts()

## Confirm `analysis` and `run` values

Before filtering, confirm the exact category labels — the data dictionary describes them
but doesn't give literal strings (e.g. `run` may read `"MRSA"` or a longer phenotype
label).

In [ ]:
print(raw["analysis"].value_counts())
print()
print(raw["run"].value_counts())

## Filter to MRSA cases/controls, for each analysis arm separately

In [ ]:
env = raw[(raw["analysis"].str.lower() == "environmental") &
          (raw["run"].astype(str).str.contains("MRSA", case=False, na=False))].copy()
pat = raw[(raw["analysis"].str.lower() == "patient") &
          (raw["run"].astype(str).str.contains("MRSA", case=False, na=False))].copy()

print("environmental MRSA subset:", env.shape)
print("patient MRSA subset:", pat.shape)

# Sanity check against the PhysioNet data dictionary's published sample sizes:
# environmental MRSA: 1,101 cases / 2,397 controls
# patient MRSA:       1,102 cases / 2,656 controls
print(env["group_binary"].value_counts())
print(pat["group_binary"].value_counts())

## Fix `age`

Ages >=90 are recorded as the literal string `'>90'` for privacy, so the column can't be
treated as numeric as-is. `clean_age` recodes it to a numeric sentinel and adds an
`age_over_90` flag so the censoring stays visible rather than silently biasing the mean
downward.

In [ ]:
raw["age"].apply(type).value_counts()

In [ ]:
env = clean_age(env)
pat = clean_age(pat)

print("environmental age_over_90 rate:", env["age_over_90"].mean().round(4))
print("patient age_over_90 rate:", pat["age_over_90"].mean().round(4))
env[["age", "age_over_90"]].describe(include="all")

## Missingness

In [ ]:
env.isna().mean().sort_values(ascending=False).head(20)

In [ ]:
pat.isna().mean().sort_values(ascending=False).head(20)

## Check matching assumptions

The environmental cohort is matched on age/sex, so case vs. control distributions of age
and sex should look near-identical there. The patient cohort is **not** matched on
age/sex — if those distributions diverge here, it confirms the model in
`04b_logreg_patient.ipynb` needs to control for age/sex explicitly rather than relying on
the match.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
sns.boxplot(data=env, x="group", y="age", ax=axes[0])
axes[0].set_title("Environmental (matched on age) — age by group")
sns.boxplot(data=pat, x="group", y="age", ax=axes[1])
axes[1].set_title("Patient (NOT matched on age) — age by group")
plt.tight_layout()

In [ ]:
print("Environmental — sex by group (row %):")
print(pd.crosstab(env["group"], env["sex"], normalize="index").round(3))
print()
print("Patient — sex by group (row %):")
print(pd.crosstab(pat["group"], pat["sex"], normalize="index").round(3))

## Class balance and key predictor distributions

In [ ]:
cp_cols = [c for c in env.columns if c.endswith("_cp")]
print("Colonization pressure columns:", cp_cols)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=env, x="group", y="MRSA_cp", ax=ax)
ax.set_title("Environmental — MRSA colonization pressure by group")
plt.tight_layout()

In [ ]:
abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]
print("Antibiotic class course-count columns:", abx_cols)

pat.groupby("group")[abx_cols].mean().T.sort_values("Case", ascending=False)

## Save cleaned MRSA subsets for downstream notebooks

In [ ]:
save_processed(env, "environmental_mrsa.csv")
save_processed(pat, "patient_mrsa.csv")

## Next steps

- `02_correlation_analysis.ipynb` — correlation structure among candidate predictors
  within each arm.
- `03_hypothesis_testing.ipynb` — formal case vs. control comparisons.
- `04a_logreg_environmental.ipynb` / `04b_logreg_patient.ipynb` — logistic regression per
  analysis arm (patient model controls for age/sex explicitly, per the imbalance checked
  above).